# SLE journal external validation and claim audit

This notebook closes five remaining journal-review risks:

1. biometric utility is evaluated over repeated gallery/probe splits;
2. the protocol is evaluated on both CelebA and an LFW-derived
   closed-set identification protocol;
3. SLE and Secret Pointer Join are placed in one runtime,
   communication, and empirical-unlinkability table;
4. security assumptions and the unlinkability argument are stated
   explicitly;
5. protected and out-of-scope compromise cases are exported as a
   machine-readable claim registry.

The LFW experiment below is a custom closed-set identification
protocol. It must not be described as the official LFW verification
protocol. The notebook uses names only to group images and never
exports them.

The embedded MP-SPDZ measurements and held-out attack measurements
come from the previously completed validated runs. They are included
to synthesize the evidence in one table. The original scaling and
security notebooks remain the executable sources for regenerating
those measurements.


In [ ]:
# CELL 1. Reproducible experiment controls
SEED = 20260722

N_IDENTITIES = 128
IMAGES_PER_ID = 6
GALLERY_IMAGES_PER_ID = 2
N_SPLITS = 20
EMBEDDING_BITS = 512

CELEBA_DATASET = 'flwrlabs/celeba'
CELEBA_SPLIT = 'train'
CELEBA_IMAGE_KEY = 'image'
CELEBA_IDENTITY_KEY = 'celeb_id'
MAX_CELEBA_STREAM_ROWS = 170_000

LFW_MIN_FACES = 6
LFW_RESIZE = 1.0

USE_SYNTHETIC_SMOKE = False
OUTPUT_DIR = '/content/sle_journal_external_validation_outputs'

print({
    'identities_per_dataset': N_IDENTITIES,
    'images_per_identity': IMAGES_PER_ID,
    'gallery_images': GALLERY_IMAGES_PER_ID,
    'randomized_splits': N_SPLITS,
    'embedding_bits': EMBEDDING_BITS,
})


## Security model and unlinkability argument

Let the secret locator for identity $i$ at epoch $e$ be

$$\ell_i^{e+1}=F_{K_{e+1}}(\ell_i^e),$$

where $F$ is a secure pseudorandom permutation evaluated inside MPC
and $K_{e+1}$ is a fresh secret epoch key. Database and index copies
are independently secret-shared, transformed separately, and checked
through a hidden multiset relation. No old-to-new pointer map is
opened.

Under hidden independent epoch keys, secure MPC/ORAM execution, and
absent persistent metadata, each new locator assignment is
computationally indistinguishable from a fresh random permutation.
The cross-epoch linking advantage is therefore bounded by the PRP and
MPC distinguishing advantages plus explicitly leaked metadata. This
is an argument under assumptions, not a complete reduction proof.

For relation integrity, define

$$\Phi(S,r)=\prod_{x\in S}(r-E(x)).$$

If two committed multisets of at most $M$ elements differ and $E$ is
injective, the polynomial difference is non-zero with degree at most
$M-1$. A uniformly sampled challenge accepts an unequal relation with
probability at most $(M-1)/|\mathbb{F}|$. Two independent challenges
reduce this bound to at most $((M-1)/|\mathbb{F}|)^2$, provided the
candidate state is fixed before challenges are sampled.

Empirical near-random AUC supports the observable-token claim but does
not replace formal review of the PRP, field encoding, commitment
order, ORAM, or MPC backend.


## Environment

Core Colab packages are left intact. `facenet-pytorch` is installed
without replacing NumPy, PyTorch, or pandas, avoiding the binary
incompatibility encountered in earlier notebooks.


In [ ]:
# CELL 2. Safe dependencies and imports
import importlib.util
import io
import json
import math
import shutil
import subprocess
import sys
from collections import Counter
from pathlib import Path


def install_if_missing(module_name, pip_args):
    if importlib.util.find_spec(module_name) is None:
        print('Installing:', ' '.join(pip_args))
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pip_args])
    else:
        print(module_name, 'is already available')


install_if_missing('facenet_pytorch', ['--no-deps', 'facenet-pytorch==2.6.0'])
install_if_missing('datasets', ['datasets'])

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Image as HFImage
from datasets import load_dataset
from facenet_pytorch import InceptionResnetV1
from IPython.display import display
from PIL import Image as PILImage
from sklearn.datasets import fetch_lfw_people
from sklearn.metrics import roc_curve
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
from tqdm.auto import tqdm

OUTPUT = Path(OUTPUT_DIR)
OUTPUT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Device:', DEVICE)
print('PyTorch:', torch.__version__)
print('NumPy:', np.__version__)


## Explicit threat model

The table distinguishes a meaningful claim from an unlimited claim.
SLE protects cross-epoch locator relations only while epoch keys,
binary biometric roots, and persistent bucket metadata remain secret.


In [ ]:
# CELL 3. Export security assumptions and compromise boundaries
threat_model_df = pd.DataFrame([
    {
        'item': 'Epoch locator tokens',
        'adversary_access': 'Observed before and after re-keying',
        'expected_property': 'Cross-epoch unlinkability near random',
        'in_scope': True,
    },
    {
        'item': 'Top-1/reject and transition accept/reject',
        'adversary_access': 'Opened protocol outputs',
        'expected_property': 'Only intended outputs are revealed',
        'in_scope': True,
    },
    {
        'item': 'Epoch keys',
        'adversary_access': 'Secret-shared; threshold not exceeded',
        'expected_property': 'At least one contribution remains unknown',
        'in_scope': True,
    },
    {
        'item': 'MPC and ORAM transcript',
        'adversary_access': 'Passive corruption within backend threshold',
        'expected_property': 'Inputs and access patterns remain hidden',
        'in_scope': True,
    },
    {
        'item': 'Binary biometric root compromise',
        'adversary_access': 'Root reconstructed by attacker',
        'expected_property': 'No unlinkability guarantee',
        'in_scope': False,
    },
    {
        'item': 'Epoch-key compromise',
        'adversary_access': 'Complete epoch key reconstructed',
        'expected_property': 'No transition secrecy guarantee',
        'in_scope': False,
    },
    {
        'item': 'Persistent bucket metadata',
        'adversary_access': 'Signatures visible across epochs',
        'expected_property': 'No unlinkability guarantee; must stay in MPC/ORAM',
        'in_scope': False,
    },
    {
        'item': 'Malicious majority or active backend break',
        'adversary_access': 'Corruption beyond passive Shamir model',
        'expected_property': 'Fault tests are not an active-security proof',
        'in_scope': False,
    },
])

assumptions_df = pd.DataFrame([
    {'id': 'A1', 'assumption': 'F is a secure PRP under a fresh hidden epoch key.'},
    {'id': 'A2', 'assumption': 'MPC privacy holds within the stated corruption threshold.'},
    {'id': 'A3', 'assumption': 'ORAM hides data-dependent index access patterns.'},
    {'id': 'A4', 'assumption': 'Candidate state is committed before fingerprint challenges.'},
    {'id': 'A5', 'assumption': 'Tuple encoding is injective in the selected field.'},
    {'id': 'A6', 'assumption': 'Epoch commit is atomic; rejected updates preserve old state.'},
    {'id': 'A7', 'assumption': 'Roots, full keys, and persistent bucket signatures are not exposed.'},
])

threat_model_df.to_csv(OUTPUT / 'threat_model.csv', index=False)
assumptions_df.to_csv(OUTPUT / 'security_assumptions.csv', index=False)
display(threat_model_df)
display(assumptions_df)


## CelebA: deterministic identity selection

Two streaming passes avoid downloading and reconstructing the full
dataset locally. Exactly six images are retained for each of 128
identities. Appearance and recognition performance are not used for
selection.


In [ ]:
# CELL 4. Stream six CelebA images for each selected identity
def image_record_to_bytes(record):
    if isinstance(record, dict):
        if record.get('bytes') is not None:
            return bytes(record['bytes'])
        if record.get('path'):
            return Path(record['path']).read_bytes()
    if isinstance(record, PILImage.Image):
        buffer = io.BytesIO()
        record.convert('RGB').save(buffer, format='JPEG', quality=95)
        return buffer.getvalue()
    raise ValueError(f'Unsupported image record: {type(record)}')


def decode_rgb(image_bytes):
    with PILImage.open(io.BytesIO(image_bytes)) as image:
        return image.convert('RGB').copy()


def make_synthetic_faces(dataset_seed):
    rng = np.random.default_rng(dataset_seed)
    result = {}
    for identity in range(N_IDENTITIES):
        base = rng.integers(30, 225, size=(1, 1, 3), dtype=np.uint8)
        result[identity] = []
        for _ in range(IMAGES_PER_ID):
            noise = rng.normal(0, 18, size=(160, 160, 3))
            array = np.clip(base + noise, 0, 255).astype(np.uint8)
            result[identity].append(PILImage.fromarray(array, mode='RGB'))
    return result


def collect_celeba_faces():
    print('CelebA pass 1/2: count identity labels')
    stream = load_dataset(
        CELEBA_DATASET,
        split=CELEBA_SPLIT,
        streaming=True,
    ).select_columns([CELEBA_IDENTITY_KEY])

    counts = Counter()
    for inspected, row in enumerate(stream, start=1):
        counts[int(row[CELEBA_IDENTITY_KEY])] += 1
        if inspected % 25_000 == 0:
            eligible_now = sum(value >= IMAGES_PER_ID for value in counts.values())
            print(f'rows={inspected:,}, eligible={eligible_now:,}')
        if inspected >= MAX_CELEBA_STREAM_ROWS:
            break

    eligible = sorted(identity for identity, count in counts.items() if count >= IMAGES_PER_ID)
    if len(eligible) < N_IDENTITIES:
        raise RuntimeError(f'Only {len(eligible)} eligible CelebA identities found.')

    rng = np.random.default_rng(SEED)
    selected = sorted(rng.choice(eligible, N_IDENTITIES, replace=False).astype(int).tolist())
    selected_set = set(selected)

    print('CelebA pass 2/2: collect selected images')
    stream = load_dataset(
        CELEBA_DATASET,
        split=CELEBA_SPLIT,
        streaming=True,
    ).select_columns([CELEBA_IMAGE_KEY, CELEBA_IDENTITY_KEY])
    stream = stream.cast_column(CELEBA_IMAGE_KEY, HFImage(decode=False))

    compressed = {identity: [] for identity in selected}
    complete = set()
    for inspected, row in enumerate(stream, start=1):
        identity = int(row[CELEBA_IDENTITY_KEY])
        if identity in selected_set and len(compressed[identity]) < IMAGES_PER_ID:
            compressed[identity].append(image_record_to_bytes(row[CELEBA_IMAGE_KEY]))
            if len(compressed[identity]) == IMAGES_PER_ID:
                complete.add(identity)
        if inspected % 25_000 == 0:
            print(f'rows={inspected:,}, complete={len(complete)}/{N_IDENTITIES}')
        if len(complete) == N_IDENTITIES or inspected >= MAX_CELEBA_STREAM_ROWS:
            break

    if len(complete) != N_IDENTITIES:
        raise RuntimeError(f'Incomplete CelebA collection: {N_IDENTITIES-len(complete)} identities missing.')
    return {
        identity: [decode_rgb(value) for value in compressed[identity]]
        for identity in selected
    }


celeba_images = (
    make_synthetic_faces(SEED) if USE_SYNTHETIC_SMOKE else collect_celeba_faces()
)
print('CelebA identities:', len(celeba_images))


## LFW: external dataset

`fetch_lfw_people` downloads the funneled LFW images. Identities with
fewer than six images are excluded, 128 eligible identities are
sampled deterministically, and exactly six images are retained.


In [ ]:
# CELL 5. Load an LFW-derived 128-identity closed-set dataset
def collect_lfw_faces():
    lfw = fetch_lfw_people(
        funneled=True,
        resize=LFW_RESIZE,
        min_faces_per_person=LFW_MIN_FACES,
        color=True,
        download_if_missing=True,
    )
    labels = np.asarray(lfw.target, dtype=np.int64)
    eligible = np.unique(labels)
    if len(eligible) < N_IDENTITIES:
        raise RuntimeError(
            f'LFW has only {len(eligible)} identities with at least {LFW_MIN_FACES} images.'
        )

    rng = np.random.default_rng(SEED + 1)
    selected = np.sort(rng.choice(eligible, N_IDENTITIES, replace=False))
    images = {}
    reported_input_range = False
    for identity in selected:
        positions = np.flatnonzero(labels == identity)
        chosen = rng.choice(positions, IMAGES_PER_ID, replace=False)
        identity_images = []
        for position in chosen:
            array = np.asarray(lfw.images[position], dtype=np.float32)
            if array.ndim == 2:
                array = np.repeat(array[..., None], 3, axis=2)

            # Depending on the scikit-learn/imageio combination, LFW
            # can be returned either in [0, 1] or [0, 255]. Converting
            # a normalized float image directly to uint8 would make it
            # nearly black and collapse every biometric template.
            if not reported_input_range:
                print(
                    'Raw LFW pixel range:',
                    float(array.min()), float(array.max()),
                    'dtype:', array.dtype,
                )
                reported_input_range = True
            if float(array.max()) <= 1.5:
                array = array * 255.0
            array = np.clip(array, 0, 255).astype(np.uint8)
            if float(array.std()) < 2.0:
                raise RuntimeError(
                    'LFW image variance collapsed during conversion; '
                    'inspect the reported pixel range.'
                )
            identity_images.append(PILImage.fromarray(array, mode='RGB'))
        images[int(identity)] = identity_images
    return images


lfw_images = (
    make_synthetic_faces(SEED + 1) if USE_SYNTHETIC_SMOKE else collect_lfw_faces()
)
print('LFW identities:', len(lfw_images))


## FaceNet/VGGFace2 embeddings

The same frozen backbone and preprocessing are used for both datasets.
Floating-point embeddings remain in memory only until all randomized
gallery/probe splits are evaluated. They are not included in the
downloadable archive.


In [ ]:
# CELL 6. Extract per-image embeddings for both datasets on GPU
facenet_transform = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])


def extract_per_identity_embeddings(images_by_identity, model, dataset_name):
    identities = np.asarray(sorted(images_by_identity), dtype=np.int64)
    tensors = []
    for identity in identities:
        samples = images_by_identity[int(identity)]
        if len(samples) != IMAGES_PER_ID:
            raise ValueError(f'{dataset_name}: identity {identity} has {len(samples)} images.')
        tensors.extend(facenet_transform(image) for image in samples)

    image_tensor = torch.stack(tensors)
    loader = DataLoader(TensorDataset(image_tensor), batch_size=64, shuffle=False)
    batches = []
    with torch.inference_mode():
        for (batch,) in tqdm(loader, desc=f'{dataset_name} FaceNet embeddings'):
            embedding = F.normalize(model(batch.to(DEVICE)), dim=1)
            batches.append(embedding.cpu())
    array = torch.cat(batches).numpy().astype(np.float32)
    return identities, array.reshape(len(identities), IMAGES_PER_ID, -1)


facenet = InceptionResnetV1(pretrained='vggface2').eval().to(DEVICE)
celeba_ids, celeba_embeddings = extract_per_identity_embeddings(
    celeba_images, facenet, 'CelebA'
)
del celeba_images

lfw_ids, lfw_embeddings = extract_per_identity_embeddings(
    lfw_images, facenet, 'LFW'
)
del lfw_images, facenet
if torch.cuda.is_available():
    torch.cuda.empty_cache()

def embedding_sanity(name, identities, embeddings):
    flat = embeddings.reshape(-1, embeddings.shape[-1])
    roots = (flat >= 0).astype(np.uint8)
    unique_roots = int(np.unique(roots, axis=0).shape[0])
    feature_variance = float(np.mean(np.var(flat, axis=0)))
    print({
        'dataset': name,
        'shape': embeddings.shape,
        'unique_binary_roots': unique_roots,
        'total_images': len(flat),
        'mean_feature_variance': feature_variance,
    })
    if unique_roots < len(identities):
        raise RuntimeError(
            f'{name} embedding collapse: only {unique_roots} unique roots '
            f'for {len(identities)} identities.'
        )
    if feature_variance <= 1e-8:
        raise RuntimeError(f'{name} embedding variance is effectively zero.')


embedding_sanity('CelebA', celeba_ids, celeba_embeddings)
embedding_sanity('LFW', lfw_ids, lfw_embeddings)


## Repeated gallery/probe protocols

Every split selects two gallery images independently for each identity;
the remaining four images are probes. Gallery embeddings are averaged
before sign binarization. Rank-1, Rank-5, EER, and TAR are recomputed
from scratch for every split.

These are repeated randomized protocols over the same identities, not
independent new subjects. The result measures sensitivity to image
assignment and should be described accordingly.


In [ ]:
# CELL 7. Metric helpers and one randomized split
def tar_at_far(fpr, tpr, target_far):
    valid = tpr[fpr <= target_far]
    return float(valid.max()) if len(valid) else 0.0


def evaluate_random_split(dataset_name, identities, embeddings, split_index):
    rng = np.random.default_rng(SEED + 10_000 * (1 if dataset_name == 'CelebA' else 2) + split_index)
    gallery_roots = []
    probe_roots = []
    probe_labels = []

    for identity_index, identity in enumerate(identities):
        gallery_positions = np.sort(
            rng.choice(IMAGES_PER_ID, GALLERY_IMAGES_PER_ID, replace=False)
        )
        probe_positions = np.asarray([
            value for value in range(IMAGES_PER_ID)
            if value not in set(gallery_positions.tolist())
        ])

        mean_gallery = embeddings[identity_index, gallery_positions].mean(axis=0)
        gallery_roots.append((mean_gallery >= 0).astype(np.uint8))
        probe_roots.extend((embeddings[identity_index, probe_positions] >= 0).astype(np.uint8))
        probe_labels.extend([int(identity)] * len(probe_positions))

    gallery_roots = np.asarray(gallery_roots, dtype=np.uint8)
    probe_roots = np.asarray(probe_roots, dtype=np.uint8)
    probe_labels = np.asarray(probe_labels, dtype=np.int64)

    distances = np.count_nonzero(
        probe_roots[:, None, :] != gallery_roots[None, :, :], axis=2
    )
    ranking = np.argsort(distances, axis=1, kind='stable')
    rank1 = float(np.mean(identities[ranking[:, 0]] == probe_labels))
    rank5 = float(np.mean([
        probe_labels[row] in identities[ranking[row, :5]]
        for row in range(len(probe_labels))
    ]))

    label_to_column = {int(label): index for index, label in enumerate(identities)}
    genuine_columns = np.asarray([label_to_column[int(label)] for label in probe_labels])
    similarities = 1.0 - distances.astype(np.float32) / EMBEDDING_BITS
    genuine = similarities[np.arange(len(probe_labels)), genuine_columns]
    impostor_mask = np.ones_like(similarities, dtype=bool)
    impostor_mask[np.arange(len(probe_labels)), genuine_columns] = False
    impostor = similarities[impostor_mask]

    verification_labels = np.concatenate([
        np.ones(len(genuine), dtype=np.uint8),
        np.zeros(len(impostor), dtype=np.uint8),
    ])
    verification_scores = np.concatenate([genuine, impostor])
    fpr, tpr, _ = roc_curve(verification_labels, verification_scores)
    fnr = 1.0 - tpr
    eer_index = int(np.argmin(np.abs(fpr - fnr)))

    return {
        'dataset': dataset_name,
        'split': split_index,
        'identities': len(identities),
        'gallery_per_identity': GALLERY_IMAGES_PER_ID,
        'probe_images': len(probe_labels),
        'rank1': rank1,
        'rank5': rank5,
        'eer': float((fpr[eer_index] + fnr[eer_index]) / 2.0),
        'tar_at_far_1e-2': tar_at_far(fpr, tpr, 1e-2),
        'tar_at_far_1e-3': tar_at_far(fpr, tpr, 1e-3),
    }


In [ ]:
# CELL 8. Run 20 protocols per dataset and summarize uncertainty
split_rows = []
for split_index in tqdm(range(N_SPLITS), desc='CelebA randomized splits'):
    split_rows.append(evaluate_random_split(
        'CelebA', celeba_ids, celeba_embeddings, split_index
    ))
for split_index in tqdm(range(N_SPLITS), desc='LFW randomized splits'):
    split_rows.append(evaluate_random_split(
        'LFW', lfw_ids, lfw_embeddings, split_index
    ))

split_metrics_df = pd.DataFrame(split_rows)
metric_names = ['rank1', 'rank5', 'eer', 'tar_at_far_1e-2', 'tar_at_far_1e-3']
summary_rows = []
bootstrap_rng = np.random.default_rng(SEED + 90)

for dataset_name, group in split_metrics_df.groupby('dataset', sort=True):
    row = {
        'dataset': dataset_name,
        'identities': int(group['identities'].iloc[0]),
        'randomized_splits': len(group),
    }
    for metric in metric_names:
        values = group[metric].to_numpy(float)
        boot = bootstrap_rng.choice(
            values, size=(5000, len(values)), replace=True
        ).mean(axis=1)
        row[f'{metric}_mean'] = float(values.mean())
        row[f'{metric}_std'] = float(values.std(ddof=1))
        row[f'{metric}_ci_low'] = float(np.quantile(boot, 0.025))
        row[f'{metric}_ci_high'] = float(np.quantile(boot, 0.975))
        row[f'{metric}_min'] = float(values.min())
        row[f'{metric}_max'] = float(values.max())
    summary_rows.append(row)

biometric_external_summary_df = pd.DataFrame(summary_rows)
split_metrics_df.to_csv(OUTPUT / 'biometric_randomized_splits.csv', index=False)
biometric_external_summary_df.to_csv(
    OUTPUT / 'biometric_external_summary.csv', index=False
)
display(biometric_external_summary_df)


In [ ]:
# CELL 9. Publication figure for cross-dataset robustness
palette = {'CelebA': '#3B6EA8', 'LFW': '#B65C45'}
fig, axes = plt.subplots(1, 3, figsize=(12.5, 4.0))

for position, metric in enumerate(['rank1', 'eer', 'tar_at_far_1e-3']):
    ax = axes[position]
    groups = []
    labels = []
    colors = []
    for dataset_name in ['CelebA', 'LFW']:
        values = split_metrics_df.loc[
            split_metrics_df['dataset'] == dataset_name, metric
        ].to_numpy()
        groups.append(values)
        labels.append(dataset_name)
        colors.append(palette[dataset_name])

    box = ax.boxplot(groups, labels=labels, patch_artist=True, widths=0.55)
    for patch, color in zip(box['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.65)
    for index, values in enumerate(groups, start=1):
        jitter = np.linspace(-0.08, 0.08, len(values))
        ax.scatter(index + jitter, values, s=14, color=colors[index - 1], alpha=0.65)
    ax.grid(axis='y', alpha=0.25)

axes[0].set_title('(a) Rank-1 identification')
axes[0].set_ylabel('Accuracy')
axes[1].set_title('(b) Verification EER')
axes[1].set_ylabel('EER')
axes[2].set_title('(c) TAR at FAR=0.1%')
axes[2].set_ylabel('TAR')
fig.tight_layout()
fig.savefig(OUTPUT / '01_external_biometric_validation.png', dpi=260, bbox_inches='tight')
plt.show()

# Explicitly remove floating-point biometric representations before
# packaging any result artifact.
del celeba_embeddings, lfw_embeddings
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Floating-point embeddings deleted; only aggregate metrics remain.')


## Joint efficiency and unlinkability evidence

The following records are embedded from the completed five-party
MP-SPDZ scaling run and the 15-cluster held-out attack run. This cell
does not pretend to remeasure MPC performance. It produces the single
comparison table requested for the paper while retaining measurement
provenance.


In [ ]:
# CELL 10. Load validated prior measurements with provenance
scaling_df = pd.DataFrame([{"mode": "full_scan", "n": 8, "secure_seconds_mean": 0.2623726, "secure_seconds_std": 0.0127039957218191, "total_seconds_mean": 0.2677492, "total_seconds_std": 0.0127306417630848, "global_mb_mean": 0.259022, "runtime_rounds_mean": 558.0, "integer_triples_mean": 289.0}, {"mode": "full_scan", "n": 16, "secure_seconds_mean": 0.3568006, "secure_seconds_std": 0.038114777859775, "total_seconds_mean": 0.3625694, "total_seconds_std": 0.0388576081070361, "global_mb_mean": 0.505598, "runtime_rounds_mean": 734.0, "integer_triples_mean": 577.0}, {"mode": "full_scan", "n": 32, "secure_seconds_mean": 0.499417, "secure_seconds_std": 0.010424794098686, "total_seconds_mean": 0.5057777999999999, "total_seconds_std": 0.0102124424943301, "global_mb_mean": 0.99875, "runtime_rounds_mean": 1087.0, "integer_triples_mean": 1153.0}, {"mode": "full_scan", "n": 64, "secure_seconds_mean": 1.0473089999999998, "secure_seconds_std": 0.3095641691200711, "total_seconds_mean": 1.0557706, "total_seconds_std": 0.3122677755145414, "global_mb_mean": 1.98505, "runtime_rounds_mean": 1792.0, "integer_triples_mean": 2305.0}, {"mode": "full_scan", "n": 128, "secure_seconds_mean": 1.491204, "secure_seconds_std": 0.0556873870818159, "total_seconds_mean": 1.501372, "total_seconds_std": 0.0555360740420134, "global_mb_mean": 3.95766, "runtime_rounds_mean": 3202.0, "integer_triples_mean": 4609.0}, {"mode": "oram", "n": 8, "secure_seconds_mean": 1.467568, "secure_seconds_std": 0.0157920223530743, "total_seconds_mean": 21.93766, "total_seconds_std": 0.5198502938346773, "global_mb_mean": 18.8053, "runtime_rounds_mean": 37398.0, "integer_triples_mean": 104310.0}, {"mode": "oram", "n": 16, "secure_seconds_mean": 1.954582, "secure_seconds_std": 0.3720575160778236, "total_seconds_mean": 22.81836, "total_seconds_std": 0.9689487953447272, "global_mb_mean": 21.4149, "runtime_rounds_mean": 38689.0, "integer_triples_mean": 119886.0}, {"mode": "oram", "n": 32, "secure_seconds_mean": 4.989346, "secure_seconds_std": 0.8505359477882166, "total_seconds_mean": 27.11882, "total_seconds_std": 0.5338122300959401, "global_mb_mean": 42.9724, "runtime_rounds_mean": 45637.0, "integer_triples_mean": 249294.0}, {"mode": "oram", "n": 64, "secure_seconds_mean": 11.8851, "secure_seconds_std": 0.1790338375838481, "total_seconds_mean": 41.04884, "total_seconds_std": 0.777442800468304, "global_mb_mean": 97.232, "runtime_rounds_mean": 67239.0, "integer_triples_mean": 579834.0}, {"mode": "oram", "n": 128, "secure_seconds_mean": 31.40586, "secure_seconds_std": 0.920703851952408, "total_seconds_mean": 84.39308, "total_seconds_std": 1.50754437314462, "global_mb_mean": 252.614, "runtime_rounds_mean": 137566.0, "integer_triples_mean": 1537326.0}, {"mode": "pointer_join", "n": 8, "secure_seconds_mean": 4.335504, "secure_seconds_std": 0.5747008920125322, "total_seconds_mean": 4.3449860000000005, "total_seconds_std": 0.5779549559697538, "global_mb_mean": 32.8195, "runtime_rounds_mean": 5986.0, "integer_triples_mean": 8213.0}, {"mode": "pointer_join", "n": 16, "secure_seconds_mean": 5.51029, "secure_seconds_std": 0.9255024324387268, "total_seconds_mean": 5.517362, "total_seconds_std": 0.9258753818522232, "global_mb_mean": 58.1533, "runtime_rounds_mean": 5994.0, "integer_triples_mean": 12589.0}, {"mode": "pointer_join", "n": 32, "secure_seconds_mean": 15.52216, "secure_seconds_std": 1.1283117933443745, "total_seconds_mean": 15.52882, "total_seconds_std": 1.1287382433496262, "global_mb_mean": 232.338, "runtime_rounds_mean": 11937.0, "integer_triples_mean": 37469.0}, {"mode": "pointer_join", "n": 64, "secure_seconds_mean": 37.73084, "secure_seconds_std": 0.2957644552680397, "total_seconds_mean": 37.74178, "total_seconds_std": 0.2966803700954928, "global_mb_mean": 701.453, "runtime_rounds_mean": 20758.0, "integer_triples_mean": 105149.0}, {"mode": "pointer_join", "n": 128, "secure_seconds_mean": 92.06196, "secure_seconds_std": 0.4505420823852052, "total_seconds_mean": 92.07522, "total_seconds_std": 0.452046620383337, "global_mb_mean": 1951.8, "runtime_rounds_mean": 34053.0, "integer_triples_mean": 285821.0}, {"mode": "sle", "n": 8, "secure_seconds_mean": 0.8538214, "secure_seconds_std": 0.0163748497794636, "total_seconds_mean": 0.8597666, "total_seconds_std": 0.016536347157096, "global_mb_mean": 9.60544, "runtime_rounds_mean": 1516.0, "integer_triples_mean": 59669.0}, {"mode": "sle", "n": 16, "secure_seconds_mean": 0.8537942, "secure_seconds_std": 0.0183250544528522, "total_seconds_mean": 0.8596358000000001, "total_seconds_std": 0.017925185736834, "global_mb_mean": 9.82252, "runtime_rounds_mean": 1524.0, "integer_triples_mean": 60973.0}, {"mode": "sle", "n": 32, "secure_seconds_mean": 1.615156, "secure_seconds_std": 0.5236855209283526, "total_seconds_mean": 1.62381, "total_seconds_std": 0.5241499785843743, "global_mb_mean": 19.6202, "runtime_rounds_mean": 2310.0, "integer_triples_mean": 121949.0}, {"mode": "sle", "n": 64, "secure_seconds_mean": 2.374914, "secure_seconds_std": 0.6701620817981274, "total_seconds_mean": 2.384252, "total_seconds_std": 0.6711106534469558, "global_mb_mean": 32.9731, "runtime_rounds_mean": 3371.0, "integer_triples_mean": 204989.0}, {"mode": "sle", "n": 128, "secure_seconds_mean": 3.21973, "secure_seconds_std": 0.5545410276525986, "total_seconds_mean": 3.233692, "total_seconds_std": 0.5554519525305498, "global_mb_mean": 50.3156, "runtime_rounds_mean": 4717.0, "integer_triples_mean": 312701.0}])
security_df = pd.DataFrame([{"condition": "SLE", "N": 32, "attacker": "Extra Trees", "chance_top1": 0.03125, "pair_auc_mean": 0.4987527826780915, "pair_auc_ci_low": 0.4895289595409106, "pair_auc_ci_high": 0.5077363570018482, "hungarian_accuracy_mean": 0.0325520833333333, "hungarian_accuracy_ci_low": 0.0265625, "hungarian_accuracy_ci_high": 0.0393229166666666, "effective_pair_auc": 0.5012472173219085}, {"condition": "SLE", "N": 32, "attacker": "Hamming baseline", "chance_top1": 0.03125, "pair_auc_mean": 0.4997920866935484, "pair_auc_ci_low": 0.4915201166624663, "pair_auc_ci_high": 0.5077821793094759, "hungarian_accuracy_mean": 0.0322916666666666, "hungarian_accuracy_ci_low": 0.0268229166666666, "hungarian_accuracy_ci_high": 0.03828125, "effective_pair_auc": 0.5002079133064515}, {"condition": "SLE", "N": 32, "attacker": "Logistic regression", "chance_top1": 0.03125, "pair_auc_mean": 0.4984427503360216, "pair_auc_ci_low": 0.487457052251344, "pair_auc_ci_high": 0.5084240394510249, "hungarian_accuracy_mean": 0.03125, "hungarian_accuracy_ci_low": 0.0270833333333333, "hungarian_accuracy_ci_high": 0.0356770833333333, "effective_pair_auc": 0.5015572496639784}, {"condition": "SLE", "N": 64, "attacker": "Extra Trees", "chance_top1": 0.015625, "pair_auc_mean": 0.504011882802166, "pair_auc_ci_low": 0.4999273818888993, "pair_auc_ci_high": 0.5081560253970836, "hungarian_accuracy_mean": 0.0162760416666666, "hungarian_accuracy_ci_low": 0.013671875, "hungarian_accuracy_ci_high": 0.0188802083333333, "effective_pair_auc": 0.504011882802166}, {"condition": "SLE", "N": 64, "attacker": "Hamming baseline", "chance_top1": 0.015625, "pair_auc_mean": 0.4979965129226604, "pair_auc_ci_low": 0.493218343018224, "pair_auc_ci_high": 0.5034035269923942, "hungarian_accuracy_mean": 0.0141927083333333, "hungarian_accuracy_ci_low": 0.0124967447916666, "hungarian_accuracy_ci_high": 0.016015625, "effective_pair_auc": 0.5020034870773395}, {"condition": "SLE", "N": 64, "attacker": "Logistic regression", "chance_top1": 0.015625, "pair_auc_mean": 0.4976500496031746, "pair_auc_ci_low": 0.4917606753394716, "pair_auc_ci_high": 0.5036182116957569, "hungarian_accuracy_mean": 0.015625, "hungarian_accuracy_ci_low": 0.0127604166666666, "hungarian_accuracy_ci_high": 0.0182291666666666, "effective_pair_auc": 0.5023499503968254}, {"condition": "SLE", "N": 128, "attacker": "Extra Trees", "chance_top1": 0.0078125, "pair_auc_mean": 0.496539404761447, "pair_auc_ci_low": 0.4909682311911596, "pair_auc_ci_high": 0.5018841021079716, "hungarian_accuracy_mean": 0.0096354166666666, "hungarian_accuracy_ci_low": 0.008203125, "hungarian_accuracy_ci_high": 0.0109375, "effective_pair_auc": 0.503460595238553}, {"condition": "SLE", "N": 128, "attacker": "Hamming baseline", "chance_top1": 0.0078125, "pair_auc_mean": 0.4984776634556728, "pair_auc_ci_low": 0.4925330902960669, "pair_auc_ci_high": 0.5041648567880546, "hungarian_accuracy_mean": 0.0080729166666666, "hungarian_accuracy_ci_low": 0.0068359375, "hungarian_accuracy_ci_high": 0.0093098958333333, "effective_pair_auc": 0.5015223365443271}, {"condition": "SLE", "N": 128, "attacker": "Logistic regression", "chance_top1": 0.0078125, "pair_auc_mean": 0.5006747829006726, "pair_auc_ci_low": 0.4965775188936649, "pair_auc_ci_high": 0.5051471618091652, "hungarian_accuracy_mean": 0.0088541666666666, "hungarian_accuracy_ci_low": 0.0078125, "hungarian_accuracy_ci_high": 0.0098958333333333, "effective_pair_auc": 0.5006747829006726}, {"condition": "Secret Pointer Join", "N": 32, "attacker": "Extra Trees", "chance_top1": 0.03125, "pair_auc_mean": 0.5006179645497312, "pair_auc_ci_low": 0.4888973375546034, "pair_auc_ci_high": 0.5122144736013104, "hungarian_accuracy_mean": 0.0223958333333333, "hungarian_accuracy_ci_low": 0.01587890625, "hungarian_accuracy_ci_high": 0.02890625, "effective_pair_auc": 0.5006179645497312}, {"condition": "Secret Pointer Join", "N": 32, "attacker": "Hamming baseline", "chance_top1": 0.03125, "pair_auc_mean": 0.4976399739583333, "pair_auc_ci_low": 0.4885616277879284, "pair_auc_ci_high": 0.5067422715053763, "hungarian_accuracy_mean": 0.0330729166666666, "hungarian_accuracy_ci_low": 0.02890625, "hungarian_accuracy_ci_high": 0.0372395833333333, "effective_pair_auc": 0.5023600260416667}, {"condition": "Secret Pointer Join", "N": 32, "attacker": "Logistic regression", "chance_top1": 0.03125, "pair_auc_mean": 0.501158224126344, "pair_auc_ci_low": 0.4935013900264616, "pair_auc_ci_high": 0.5089911544963878, "hungarian_accuracy_mean": 0.0296875, "hungarian_accuracy_ci_low": 0.025, "hungarian_accuracy_ci_high": 0.0346354166666666, "effective_pair_auc": 0.501158224126344}, {"condition": "Secret Pointer Join", "N": 64, "attacker": "Extra Trees", "chance_top1": 0.015625, "pair_auc_mean": 0.5001655699714782, "pair_auc_ci_low": 0.494903519241898, "pair_auc_ci_high": 0.5058464357083436, "hungarian_accuracy_mean": 0.0153645833333333, "hungarian_accuracy_ci_low": 0.0130208333333333, "hungarian_accuracy_ci_high": 0.0177083333333333, "effective_pair_auc": 0.5001655699714782}, {"condition": "Secret Pointer Join", "N": 64, "attacker": "Hamming baseline", "chance_top1": 0.015625, "pair_auc_mean": 0.4950109734106315, "pair_auc_ci_low": 0.4871155140387318, "pair_auc_ci_high": 0.5039218474695922, "hungarian_accuracy_mean": 0.0140625, "hungarian_accuracy_ci_low": 0.0118489583333333, "hungarian_accuracy_ci_high": 0.016279296875, "effective_pair_auc": 0.5049890265893684}, {"condition": "Secret Pointer Join", "N": 64, "attacker": "Logistic regression", "chance_top1": 0.015625, "pair_auc_mean": 0.5024519985945768, "pair_auc_ci_low": 0.4955999828520275, "pair_auc_ci_high": 0.5089711410280258, "hungarian_accuracy_mean": 0.0171875, "hungarian_accuracy_ci_low": 0.0141927083333333, "hungarian_accuracy_ci_high": 0.0200553385416666, "effective_pair_auc": 0.5024519985945768}, {"condition": "Secret Pointer Join", "N": 128, "attacker": "Extra Trees", "chance_top1": 0.0078125, "pair_auc_mean": 0.4985802477738989, "pair_auc_ci_low": 0.4939448419205473, "pair_auc_ci_high": 0.5034311533850322, "hungarian_accuracy_mean": 0.0078776041666666, "hungarian_accuracy_ci_low": 0.0064453125, "hungarian_accuracy_ci_high": 0.0092447916666666, "effective_pair_auc": 0.501419752226101}, {"condition": "Secret Pointer Join", "N": 128, "attacker": "Hamming baseline", "chance_top1": 0.0078125, "pair_auc_mean": 0.4991086764598455, "pair_auc_ci_low": 0.4939411985342271, "pair_auc_ci_high": 0.5048233640976152, "hungarian_accuracy_mean": 0.0083333333333333, "hungarian_accuracy_ci_low": 0.0069010416666666, "hungarian_accuracy_ci_high": 0.0097005208333333, "effective_pair_auc": 0.5008913235401544}, {"condition": "Secret Pointer Join", "N": 128, "attacker": "Logistic regression", "chance_top1": 0.0078125, "pair_auc_mean": 0.5013798030342643, "pair_auc_ci_low": 0.4987320622121255, "pair_auc_ci_high": 0.5040654370165248, "hungarian_accuracy_mean": 0.0067057291666666, "hungarian_accuracy_ci_low": 0.0050130208333333, "hungarian_accuracy_ci_high": 0.0083333333333333, "effective_pair_auc": 0.5013798030342643}, {"condition": "Static locator", "N": 32, "attacker": "Hamming baseline", "chance_top1": 0.03125, "pair_auc_mean": 1.0, "pair_auc_ci_low": 1.0, "pair_auc_ci_high": 1.0, "hungarian_accuracy_mean": 1.0, "hungarian_accuracy_ci_low": 1.0, "hungarian_accuracy_ci_high": 1.0, "effective_pair_auc": 1.0}, {"condition": "Static locator", "N": 64, "attacker": "Hamming baseline", "chance_top1": 0.015625, "pair_auc_mean": 1.0, "pair_auc_ci_low": 1.0, "pair_auc_ci_high": 1.0, "hungarian_accuracy_mean": 1.0, "hungarian_accuracy_ci_low": 1.0, "hungarian_accuracy_ci_high": 1.0, "effective_pair_auc": 1.0}, {"condition": "Static locator", "N": 128, "attacker": "Hamming baseline", "chance_top1": 0.0078125, "pair_auc_mean": 1.0, "pair_auc_ci_low": 1.0, "pair_auc_ci_high": 1.0, "hungarian_accuracy_mean": 1.0, "hungarian_accuracy_ci_low": 1.0, "hungarian_accuracy_ci_high": 1.0, "effective_pair_auc": 1.0}])
bucket_ablation_df = pd.DataFrame([{"visible_fraction": 0.0, "pair_auc_mean": 0.5028431169004265, "pair_auc_std": 0.023664648432926, "hungarian_accuracy_mean": 0.0071614583333333, "hungarian_accuracy_std": 0.006791258983336, "chance_top1": 0.0078125}, {"visible_fraction": 0.02, "pair_auc_mean": 0.5128719540092889, "pair_auc_std": 0.007726269292343, "hungarian_accuracy_mean": 0.0322916666666666, "hungarian_accuracy_std": 0.0165551197446556, "chance_top1": 0.0078125}, {"visible_fraction": 0.05, "pair_auc_mean": 0.5754831709573901, "pair_auc_std": 0.0159062731722741, "hungarian_accuracy_mean": 0.1442708333333333, "hungarian_accuracy_std": 0.0323068979478705, "chance_top1": 0.0078125}, {"visible_fraction": 0.1, "pair_auc_mean": 0.7345084323032008, "pair_auc_std": 0.0207848535515039, "hungarian_accuracy_mean": 0.406640625, "hungarian_accuracy_std": 0.0487450299305378, "chance_top1": 0.0078125}, {"visible_fraction": 0.25, "pair_auc_mean": 0.991921969035792, "pair_auc_std": 0.0053823797788883, "hungarian_accuracy_mean": 0.9872395833333332, "hungarian_accuracy_std": 0.017378243077209, "chance_top1": 0.0078125}, {"visible_fraction": 0.5, "pair_auc_mean": 1.0, "pair_auc_std": 1.4453872652178552e-17, "hungarian_accuracy_mean": 1.0, "hungarian_accuracy_std": 0.0, "chance_top1": 0.0078125}, {"visible_fraction": 1.0, "pair_auc_mean": 1.0, "pair_auc_std": 0.0, "hungarian_accuracy_mean": 1.0, "hungarian_accuracy_std": 0.0, "chance_top1": 0.0078125}])

scaling_df['measurement_source'] = 'validated prior five-party MP-SPDZ run'
security_df['measurement_source'] = '15 held-out run clusters; 3000 cluster bootstraps'
bucket_ablation_df['measurement_source'] = 'real-root N=128 metadata ablation'

scaling_df.to_csv(OUTPUT / 'validated_mpc_scaling_reference.csv', index=False)
security_df.to_csv(OUTPUT / 'validated_security_reference.csv', index=False)
bucket_ablation_df.to_csv(OUTPUT / 'validated_bucket_ablation_reference.csv', index=False)
print('Embedded scaling rows:', len(scaling_df))
print('Embedded security rows:', len(security_df))


In [ ]:
# CELL 11. Build one runtime, communication, and unlinkability table
comparison_rows = []
for n_value in [32, 64, 128]:
    sle = scaling_df[(scaling_df['mode'] == 'sle') & (scaling_df['n'] == n_value)].iloc[0]
    pointer = scaling_df[
        (scaling_df['mode'] == 'pointer_join') & (scaling_df['n'] == n_value)
    ].iloc[0]

    sle_security = security_df[
        (security_df['condition'] == 'SLE')
        & (security_df['N'] == n_value)
        & (security_df['attacker'] == 'Extra Trees')
    ].iloc[0]
    pointer_security = security_df[
        (security_df['condition'] == 'Secret Pointer Join')
        & (security_df['N'] == n_value)
        & (security_df['attacker'] == 'Extra Trees')
    ].iloc[0]

    comparison_rows.append({
        'N': n_value,
        'sle_total_seconds': float(sle['total_seconds_mean']),
        'pointer_join_total_seconds': float(pointer['total_seconds_mean']),
        'runtime_speedup_pointer_over_sle': float(
            pointer['total_seconds_mean'] / sle['total_seconds_mean']
        ),
        'sle_global_mb': float(sle['global_mb_mean']),
        'pointer_join_global_mb': float(pointer['global_mb_mean']),
        'communication_reduction_pointer_over_sle': float(
            pointer['global_mb_mean'] / sle['global_mb_mean']
        ),
        'sle_effective_auc': float(sle_security['effective_pair_auc']),
        'pointer_join_effective_auc': float(pointer_security['effective_pair_auc']),
        'absolute_auc_gap': float(abs(
            sle_security['effective_pair_auc'] - pointer_security['effective_pair_auc']
        )),
        'sle_global_linking_accuracy': float(sle_security['hungarian_accuracy_mean']),
        'pointer_join_global_linking_accuracy': float(
            pointer_security['hungarian_accuracy_mean']
        ),
        'chance_linking_accuracy': float(sle_security['chance_top1']),
    })

joint_comparison_df = pd.DataFrame(comparison_rows)
joint_comparison_df.to_csv(
    OUTPUT / 'joint_efficiency_unlinkability_comparison.csv', index=False
)
display(joint_comparison_df.round(4))


In [ ]:
# CELL 12. Publication figure for the direct transition baseline
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.9))
x = joint_comparison_df['N'].to_numpy()

axes[0].plot(x, joint_comparison_df['sle_total_seconds'], marker='o', label='SLE')
axes[0].plot(
    x, joint_comparison_df['pointer_join_total_seconds'], marker='s', label='Secret Pointer Join'
)
axes[0].set_yscale('log')
axes[0].set_title('(a) Transition runtime')
axes[0].set_ylabel('Total seconds, log scale')

axes[1].plot(x, joint_comparison_df['sle_global_mb'], marker='o', label='SLE')
axes[1].plot(
    x, joint_comparison_df['pointer_join_global_mb'], marker='s', label='Secret Pointer Join'
)
axes[1].set_yscale('log')
axes[1].set_title('(b) Global communication')
axes[1].set_ylabel('MB, log scale')

axes[2].plot(x, joint_comparison_df['sle_effective_auc'], marker='o', label='SLE')
axes[2].plot(
    x, joint_comparison_df['pointer_join_effective_auc'], marker='s', label='Secret Pointer Join'
)
axes[2].axhline(0.5, color='black', linestyle=':', linewidth=1, label='Random')
axes[2].set_ylim(0.48, 0.53)
axes[2].set_title('(c) Held-out linkability')
axes[2].set_ylabel('Effective AUC')

for ax in axes:
    ax.set_xlabel('Enrolled records, N')
    ax.grid(alpha=0.25)
handles, labels = axes[2].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=3, frameon=False)
fig.tight_layout(rect=[0, 0.12, 1, 1])
fig.savefig(OUTPUT / '02_joint_efficiency_unlinkability.png', dpi=260, bbox_inches='tight')
plt.show()


## Claim registry

The registry below prevents the paper from silently turning an
empirical result into a cryptographic theorem. Each statement records
its evidence and the wording that should not be used.


In [ ]:
# CELL 13. Generate paper-safe claims and limitations
external_datasets = sorted(split_metrics_df['dataset'].unique().tolist())
minimum_external_rank1 = float(
    biometric_external_summary_df['rank1_mean'].min()
)
maximum_external_eer = float(
    biometric_external_summary_df['eer_mean'].max()
)
chance_rank1 = 1.0 / N_IDENTITIES
external_pipeline_valid = bool(
    minimum_external_rank1 > 5.0 * chance_rank1
    and maximum_external_eer < 0.45
)
max_auc_gap = float(joint_comparison_df['absolute_auc_gap'].max())
max_bucket_safe_auc = float(
    bucket_ablation_df.loc[
        bucket_ablation_df['visible_fraction'] == 0.0, 'pair_auc_mean'
    ].iloc[0]
)

claim_registry_df = pd.DataFrame([
    {
        'claim_id': 'C1',
        'status': 'supported' if external_pipeline_valid else 'failed_pipeline_sanity',
        'supported_wording': (
            (
                f'Biometric utility was evaluated on {len(external_datasets)} datasets '
                f'with {N_SPLITS} randomized gallery/probe assignments; the lower mean '
                f'Rank-1 was {minimum_external_rank1:.3f}.'
            ) if external_pipeline_valid else (
                'No cross-dataset utility claim is permitted because at least one '
                'dataset failed the biometric pipeline sanity threshold.'
            )
        ),
        'evidence': 'biometric_external_summary.csv',
        'forbidden_overclaim': 'The method generalizes to all biometric domains.',
    },
    {
        'claim_id': 'C2',
        'status': 'supported',
        'supported_wording': (
            'SLE retained near-random held-out linkability under the tested observable-token attacks.'
        ),
        'evidence': 'validated_security_reference.csv',
        'forbidden_overclaim': 'SLE is cryptographically unlinkable against every adversary.',
    },
    {
        'claim_id': 'C3',
        'status': 'supported',
        'supported_wording': (
            'Under the measured five-party MP-SPDZ configuration, SLE reduced transition '
            'runtime and communication relative to Secret Pointer Join.'
        ),
        'evidence': 'joint_efficiency_unlinkability_comparison.csv',
        'forbidden_overclaim': 'SLE is universally faster on every network and MPC backend.',
    },
    {
        'claim_id': 'C4',
        'status': 'supported',
        'supported_wording': (
            f'The maximum effective-AUC gap from Secret Pointer Join was {max_auc_gap:.4f} '
            'under the evaluated attacker.'
        ),
        'evidence': 'joint_efficiency_unlinkability_comparison.csv',
        'forbidden_overclaim': 'SLE provides stronger cryptographic security than Secret Pointer Join.',
    },
    {
        'claim_id': 'C5',
        'status': 'supported',
        'supported_wording': (
            f'With no persistent bucket signature exposed, mean metadata-attack AUC was '
            f'{max_bucket_safe_auc:.3f}; leakage increased rapidly when metadata was revealed.'
        ),
        'evidence': 'validated_bucket_ablation_reference.csv',
        'forbidden_overclaim': 'Public bucket metadata is safe.',
    },
    {
        'claim_id': 'C6',
        'status': 'supported',
        'supported_wording': (
            'The current backend and experiments support a passive-threshold empirical claim.'
        ),
        'evidence': 'threat_model.csv; security_assumptions.csv',
        'forbidden_overclaim': 'Fault injection establishes malicious MPC security.',
    },
])
claim_registry_df.to_csv(OUTPUT / 'claim_registry.csv', index=False)
display(claim_registry_df)


In [ ]:
# CELL 14. Falsifiable journal-readiness checks
lfw_count = int(split_metrics_df.loc[split_metrics_df['dataset'] == 'LFW', 'identities'].iloc[0])
celeba_count = int(split_metrics_df.loc[split_metrics_df['dataset'] == 'CelebA', 'identities'].iloc[0])
n128 = joint_comparison_df[joint_comparison_df['N'] == 128].iloc[0]

readiness_df = pd.DataFrame([
    {
        'criterion': 'Two real face datasets evaluated',
        'value': len(external_datasets),
        'threshold': '>= 2',
        'passed': len(external_datasets) >= 2 and not USE_SYNTHETIC_SMOKE,
    },
    {
        'criterion': 'At least 128 identities on each dataset',
        'value': min(lfw_count, celeba_count),
        'threshold': '>= 128',
        'passed': min(lfw_count, celeba_count) >= 128,
    },
    {
        'criterion': 'Repeated gallery/probe assignments',
        'value': int(split_metrics_df.groupby('dataset')['split'].nunique().min()),
        'threshold': '>= 20',
        'passed': split_metrics_df.groupby('dataset')['split'].nunique().min() >= 20,
    },
    {
        'criterion': 'No dataset collapsed to chance-level identification',
        'value': minimum_external_rank1,
        'threshold': f'> {5.0 * chance_rank1:.4f} (5x chance)',
        'passed': minimum_external_rank1 > 5.0 * chance_rank1,
    },
    {
        'criterion': 'No dataset has random verification behavior',
        'value': maximum_external_eer,
        'threshold': '< 0.45',
        'passed': maximum_external_eer < 0.45,
    },
    {
        'criterion': 'SLE faster than pointer join at N=128',
        'value': float(n128['runtime_speedup_pointer_over_sle']),
        'threshold': '> 1',
        'passed': n128['runtime_speedup_pointer_over_sle'] > 1,
    },
    {
        'criterion': 'SLE communicates less than pointer join at N=128',
        'value': float(n128['communication_reduction_pointer_over_sle']),
        'threshold': '> 1',
        'passed': n128['communication_reduction_pointer_over_sle'] > 1,
    },
    {
        'criterion': 'Threat model and forbidden overclaims exported',
        'value': len(claim_registry_df),
        'threshold': '>= 5 claims',
        'passed': len(claim_registry_df) >= 5,
    },
])
readiness_df.to_csv(OUTPUT / 'journal_readiness_checks.csv', index=False)
display(readiness_df)


In [ ]:
# CELL 15. Save a concise paper-ready evidence summary and ZIP archive
summary_lines = [
    '# SLE journal evidence summary',
    '',
    '## External biometric validation',
]
if not external_pipeline_valid:
    summary_lines.extend([
        '- INVALID EXTERNAL BIOMETRIC RUN: at least one dataset failed sanity checks.',
        '- Do not use cross-dataset utility values in the paper until corrected.',
    ])
for _, row in biometric_external_summary_df.iterrows():
    summary_lines.append(
        f"- {row['dataset']}: Rank-1={row['rank1_mean']:.4f} "
        f"(95% split-bootstrap CI {row['rank1_ci_low']:.4f}-"
        f"{row['rank1_ci_high']:.4f}), EER={row['eer_mean']:.4f}."
    )
summary_lines.extend([
    '',
    '## Direct transition comparison at N=128',
    f"- Runtime speedup: {n128['runtime_speedup_pointer_over_sle']:.2f}x.",
    f"- Communication reduction: {n128['communication_reduction_pointer_over_sle']:.2f}x.",
    f"- Effective-AUC gap: {n128['absolute_auc_gap']:.4f}.",
    '',
    '## Claim boundary',
    '- Empirical unlinkability is claimed only for observable locator tokens.',
    '- Root, complete key, and persistent metadata compromise are out of scope.',
    '- The current passive backend and fault tests are not an active-security proof.',
    '- LFW results use a custom closed-set protocol, not the official LFW protocol.',
])
(OUTPUT / 'PAPER_EVIDENCE_SUMMARY.md').write_text(
    '\n'.join(summary_lines) + '\n', encoding='utf-8'
)

run_metadata = {
    'seed': SEED,
    'datasets': external_datasets,
    'identities_per_dataset': N_IDENTITIES,
    'images_per_identity': IMAGES_PER_ID,
    'gallery_images_per_identity': GALLERY_IMAGES_PER_ID,
    'randomized_splits': N_SPLITS,
    'backbone': 'FaceNet InceptionResnetV1 pretrained on VGGFace2',
    'embedded_reference_sources': {
        'scaling': 'validated prior five-party MP-SPDZ scaling run',
        'security': '15 held-out run-cluster attack evaluation',
    },
    'synthetic_smoke': USE_SYNTHETIC_SMOKE,
}
(OUTPUT / 'run_metadata.json').write_text(
    json.dumps(run_metadata, indent=2), encoding='utf-8'
)

archive_path = shutil.make_archive(
    '/content/sle_journal_external_validation_results', 'zip', OUTPUT
)
print('Saved archive:', archive_path)
print('Files:')
for path in sorted(OUTPUT.iterdir()):
    print(' ', path.name)

try:
    from google.colab import files
    files.download(archive_path)
except Exception as exc:
    print('Automatic Colab download unavailable:', exc)


## Interpretation rules

A successful run supports a bounded journal claim: SLE preserves useful
biometric behavior across two evaluated datasets and repeated image
assignments, while its measured transition cost is lower than the
direct private pointer-translation baseline at similar empirical
locator unlinkability.

It does not establish universal biometric generalization, security
after root/key/metadata compromise, malicious-MPC security, or a full
cryptographic unlinkability proof. Those boundaries belong in the
methods, threat-model, limitations, and conclusion sections.
